# Export fragment files to be processed by chrombpnet

In [1]:
suppressPackageStartupMessages(library(ArchR))
suppressPackageStartupMessages(library(parallel))



                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .______      
          /   \     |   _ 

In [2]:
here::i_am("atac/chrombpnet/export_fragments.ipynb")

source(here::here("settings.R"))
source(here::here("utils.R"))


here() starts at /rds/project/rds-SDzz0CATGms/users/bt392/09_Eomes_invitro_blood/code

Warning message:
“package ‘rtracklayer’ was built under R version 4.2.3”


In [6]:
ArchRProject = loadArchRProject(io$archR.directory)
meta = fread(paste0(io$basedir, '/results/rna_atac/clustering/metadata_celltype_annotated_v2.txt.gz'))

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
           ___      .______        ______  __    __  .____

In [14]:
outdir = paste0(io$basedir, '/results/atac/chrombpnet/')
dir.create(outdir, recursive = T)

Warning message in dir.create(outdir, recursive = T):
“'/rds/project/rds-SDzz0CATGms/users/bt392/09_Eomes_invitro_blood/results/atac/chrombpnet' already exists”


In [15]:
meta_WT = meta %>% .[genotype == 'WT']

In [16]:
unique(meta_WT$celltype_v2)

[1] "Primitive_Streak"    "Early_Mes_EOi"       "Early_Mes_EOd"      
 [4] "PGC"                 "Posterior_Mes"       "HE_Precursor"       
 [7] "HE"                  "Mesenchyme"          "Allantois_Precursor"
[10] "Blood_Progenitor"    "Endothelium"         "Allantois"

In [22]:
chrs = c('chr1','chr10','chr11','chr12','chr13','chr14','chr15','chr16','chr17','chr18','chr19','chr2','chr3','chr4','chr5','chr6','chr7','chr8','chr9')
# exclude samples
# samples_incl = unique(meta$sample)[!unique(meta$sample) %in% c('E8.5_CRISPR_T_KO','E8.5_CRISPR_T_WT')]
# meta = meta[sample %in% samples_incl]

# Cell types of interest
celltypes_keep = c('Primitive_Streak', 'Early_Mes_EOi')

# Export fragments of celltype cells in subset of samples
mclapply(unique(meta_WT$sample), function(x){
    # List cells per sample
    message(x)
    # filter right samples
    tmp_meta = meta[sample == x]  
    # Keep relevant cell types
    celltypes = tmp_meta %>%
        .[, .N, by = 'celltype_v2'] %>% 
        unique(by = 'celltype_v2') %>%
        .[N > 50] %>% 
        .$celltype_v2
    
    celltypes = celltypes[celltypes %in% celltypes_keep]
    
    # Should have put an if statement to only load a file if any cell type is actually present....
    # Now it will load every file, even the ones where we won't save anything
    
    # if(length(celltypes) > 0){
    
    # Load fragment file
    file = sprintf('%s/original/%s/outs/atac_fragments.tsv.gz', io$basedir, x)
    fragment = suppressWarnings(fread(file,
                                        tmpdir = '/rds/project/rds-SDzz0CATGms/users/bt392/software/tmp'))
    message('before filtering:')
    message(nrow(fragment))
    
    mclapply(celltypes, function(i){
        message(i)
        # Only run if file doesn't exist yet
        if(!file.exists(sprintf('%s/%s_%s.tsv.gz', outdir, i, x))){
            # determine cells
            tmp_cell = tmp_meta[celltype_v2 == i, barcode]
            
            # Filter right cells & chrs from fragment file
            fragment = fragment %>% 
                setnames(c('chr', 'start', 'end', 'cell', 'reads')) %>%
                # .[chr == 'chr1'] %>%
                .[cell %in% tmp_cell] %>% 
                .[chr %in% chrs]

            message('after filtering:')
            message(nrow(fragment))

            fwrite(fragment, sprintf('%s/%s_%s.tsv.gz', outdir, i, x), sep = '\t')
        }
    }, mc.cores = 1)
}, mc.cores = 1)

1A_Eo_DEG_G9_day3

before filtering:

175376336

Primitive_Streak

after filtering:

124314666

1B_Eo_DEG_G9_day3

before filtering:

147480062

Primitive_Streak

after filtering:

99357185

2_Eo_DEG_G9_day3_5_VC

before filtering:

301618929

Early_Mes_EOi

after filtering:

34762265

Primitive_Streak

after filtering:

21703082

2_Eo_DEG_G9_day4_VC

before filtering:

184863381

Early_Mes_EOi

after filtering:

2515827

Primitive_Streak

after filtering:

2082994

2_Eo_DEG_G9_day5_VC

before filtering:

32380584

rv_eo_deg_day3_5_control

before filtering:

234284379

Primitive_Streak

after filtering:

14675780

Early_Mes_EOi

after filtering:

19724714

rv_eo_deg_day4_5_control

before filtering:

73281386

rv_eo_deg_day4_control

before filtering:

176167120



[[1]]
[[1]][[1]]
NULL


[[2]]
[[2]][[1]]
NULL


[[3]]
[[3]][[1]]
NULL

[[3]][[2]]
NULL


[[4]]
[[4]][[1]]
NULL

[[4]][[2]]
NULL


[[5]]
list()

[[6]]
[[6]][[1]]
NULL

[[6]][[2]]
NULL


[[7]]
list()

[[8]]
list()

In [23]:
celltypes_keep = c('Primitive_Streak', 'Early_Mes_EOi')

lapply(celltypes_keep, function(i){    
    message(i)
    
    # List files per cell type
    files = list.files(outdir, pattern = i)
    # overwrite combined fragment file if it exists
    if(length(grep('fragment', files))>0){
        files = files[-grep('fragment', files)] 
    }
    message(files)
    # Load in the separate fragment files per cell type
    tmp = mclapply(files, function(x){
        tmp_frag = fread(sprintf('%s/%s', outdir, x), sep = '\t',
                                        tmpdir = '/rds/project/rds-SDzz0CATGms/users/bt392/software/tmp')
        return(tmp_frag)
    }, mc.cores = 8) %>% rbindlist() %>% 
    .[order(chr, start)] %>% 
    .[,reads := 1]

    fwrite(tmp, sprintf('%s/%s_fragments.tsv.gz', outdir, i), col.names = F, sep = '\t')
})

Primitive_Streak

Primitive_Streak_1A_Eo_DEG_G9_day3.tsv.gzPrimitive_Streak_1B_Eo_DEG_G9_day3.tsv.gzPrimitive_Streak_2_Eo_DEG_G9_day3_5_VC.tsv.gzPrimitive_Streak_2_Eo_DEG_G9_day4_VC.tsv.gzPrimitive_Streak_rv_eo_deg_day3_5_control.tsv.gz

Early_Mes_EOi

Early_Mes_EOi_2_Eo_DEG_G9_day3_5_VC.tsv.gzEarly_Mes_EOi_2_Eo_DEG_G9_day4_VC.tsv.gzEarly_Mes_EOi_rv_eo_deg_day3_5_control.tsv.gz



[[1]]
NULL

[[2]]
NULL

In [24]:
peaks = as.data.table(ArchRProject@peakSet) %>%
    .[, middle := start + (end - start) / 2] %>%
    .[,`:=`(start = middle - 1057, 
            end = middle + 1057,
            column4 = '.',
            column5 = '.',
            column6 = '.',
            column7 = '.',
            column8 = '.',
            column9 = '.',
            column10 = 1057)] %>% 
    .[,.(seqnames, 
         start, 
         end, 
         column4,
         column5,
         column6,
         column7,
         column8,
         column9,
         column10)]

nrow(peaks)
head(peaks)

[1] 234908

seqnames,start,end,column4,column5,column6,column7,column8,column9,column10
<fct>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>
chr1,3002713,3004827,.,.,.,.,.,.,1057
chr1,3034841,3036955,.,.,.,.,.,.,1057
chr1,3061882,3063996,.,.,.,.,.,.,1057
chr1,3190830,3192944,.,.,.,.,.,.,1057
chr1,3262833,3264947,.,.,.,.,.,.,1057
chr1,3482191,3484305,.,.,.,.,.,.,1057


In [25]:
peaks.gr = GenomicRanges::makeGRangesFromDataFrame(peaks, keep.extra.columns = T)
overlaps = GenomicRanges::findOverlaps(peaks.gr, ArchRProject@genomeAnnotation$blacklist)

peaks.gr = peaks.gr[-queryHits(overlaps)]

In [26]:
peaks = as.data.table(peaks.gr)[,`:=`(width=NULL, strand=NULL)]
fwrite(peaks, sprintf('%s/all_peaks.bed', outdir), col.names = F, sep = '\t')

In [27]:
chromSizes = as.data.table(ArchRProject@genomeAnnotation$chromSizes) %>%
    .[,.(seqnames, end)]
fwrite(chromSizes, sprintf('%s/mm10.chrom.sizes', outdir), col.names = F, sep = '\t')

In [28]:
fwrite(as.data.table(ArchRProject@genomeAnnotation$blacklist) %>%
           .[,.(seqnames, start, end)],
       sprintf('%s/blacklist.bed.gz', outdir), col.names = F, sep = '\t')

In [ ]:
# Get background peak set using chrombpnet

In [29]:
# Run the output of the following code from the CLI in the right folder with the conda chrombpnet2 environment active
writeLines(paste0(
sapply(celltypes_keep, function(celltype){
    print(sprintf('sbatch run_chrombpnet_args.sh %s', 
                   celltype))
})),
           "test.sh")

[1] "sbatch run_chrombpnet_args.sh Primitive_Streak"
[1] "sbatch run_chrombpnet_args.sh Early_Mes_EOi"
